# Reconhecimento de Gestos Customizados

Este notebook utiliza o modelo treinado `models/gesture_model.pkl` para reconhecer gestos em tempo real a partir dos landmarks da mão detectados pelo MediaPipe.

### Pré-requisitos
Certifique-se de que o modelo foi treinado no notebook anterior (`01.2- train_gesture_model.ipynb`).

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import pickle
import time

## Carregamento do Modelo Customizado e Configuração do MediaPipe

Carregamos o classificador treinado e configuramos o MediaPipe para extrair os pontos das mãos.

In [4]:
import mediapipe as mp

# Configuração do MediaPipe Gesture Recognizer (.task)
model_path = 'models/gesture_recognizer.task'

# We read the model as a buffer to avoid path encoding issues on Windows with non-ASCII characters
with open(model_path, 'rb') as f:
    model_data = f.read()

BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_buffer=model_data),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

recognizer = GestureRecognizer.create_from_options(options)
print("MediaPipe Gesture Recognizer carregado com sucesso!")

MediaPipe Gesture Recognizer carregado com sucesso!


## Funções de Visualização

Função para desenhar os landmarks e a predição do nosso modelo customizado.

In [5]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Polegar
    (0, 5), (5, 6), (6, 7), (7, 8),      # Indicador
    (5, 9), (9, 10), (10, 11), (11, 12),  # Médio
    (9, 13), (13, 14), (14, 15), (15, 16), # Anelar
    (0, 17), (17, 18), (18, 19), (19, 20), # Mindinho
    (5, 9), (9, 13), (13, 17)             # Base da palma
]

def draw_custom_results(image, hand_landmarks, gesture_name):
    h, w, _ = image.shape
    points = []
    for lm in hand_landmarks:
        points.append((int(lm.x * w), int(lm.y * h)))

    # Desenha conexões e pontos
    for start, end in HAND_CONNECTIONS:
        cv2.line(image, points[start], points[end], (0, 255, 0), 2)
    for p in points:
        cv2.circle(image, p, 5, (255, 0, 255), -1)

    # Exibe o nome do gesto customizado com contorno para contraste
    if gesture_name:
        avg_x = sum([p[0] for p in points]) // len(points)
        avg_y = min([p[1] for p in points]) - 30
        cv2.putText(image, gesture_name, (avg_x - 50, avg_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(image, gesture_name, (avg_x - 50, avg_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

    return image

## Loop Principal de Reconhecimento

In [6]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao abrir webcam.")
else:
    print("Webcam pronta. Reconhecendo gestos customizados...")
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(time.time() * 1000)

        # Detecta landmarks
        result = recognizer.recognize_for_video(mp_image, timestamp_ms)

        # Itera sobre todas as mãos detectadas (MediaPipe Tasks permite múltiplas)
        if result.hand_landmarks:
            for idx, hand_landmarks in enumerate(result.hand_landmarks):
                # Pega a categoria do gesto reconhecido pelo MediaPipe (.task)
                if result.gestures and len(result.gestures) > idx:
                    gesture = result.gestures[idx][0]
                    gesture_name = f"{gesture.category_name} ({gesture.score:.2f})"
                else:
                    gesture_name = "Desconhecido"
                
                # Desenha resultados
                frame = draw_custom_results(frame, hand_landmarks, gesture_name)
        cv2.imshow('Custom Gesture Recognition', frame)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    recognizer.close()
    print("Encerrado.")

Webcam pronta. Reconhecendo gestos customizados...


c:\Users\Luís Felipe\Documents\GitHub\computer-vision\03- real-time-recog\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Encerrado.
